In [1]:
!pip install -q transformers datasets evaluate sacrebleu sentencepiece accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 99.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.2

In [2]:
import os, time, random
import numpy as np
import pandas as pd
import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    EarlyStoppingCallback,
    set_seed)
import evaluate

In [3]:
SEED = 42
set_seed(SEED)
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

In [4]:
DATASET_PATH      = "/kaggle/input/datasets/phuc2009/mtet-v2-290k-dataset"
MODEL_CHECKPOINT  = "Helsinki-NLP/opus-mt-en-vi"
OUTPUT_DIR        = "/kaggle/working/marianmt-mtet-model-exp02"
IELTS_PARQUET_PATH    = "/kaggle/input/datasets/phuc2009/academic-dataset/ielts_academic_clean.parquet"

In [5]:
MAX_INPUT_LENGTH  = 128
MAX_TARGET_LENGTH = 128
SOURCE_LANGUAGE   = "en"
TARGET_LANGUAGE   = "vi"

In [6]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE} | GPUs visible: {torch.cuda.device_count()}")

Device: cuda | GPUs visible: 2


In [7]:
raw_datasets = load_dataset(
    "parquet",
    data_files={
        "train":      f"{DATASET_PATH}/train.parquet",
        "validation": f"{DATASET_PATH}/valid.parquet",
        "test":       f"{DATASET_PATH}/test.parquet",
    },
)
print("Columns:", raw_datasets["train"].column_names)
print(f"train: {len(raw_datasets['train']):,} | "
      f"valid: {len(raw_datasets['validation']):,} | "
      f"test: {len(raw_datasets['test']):,}")

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

Columns: ['vi', 'en']
train: 290,000 | valid: 15,000 | test: 25,000


In [8]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT)
model     = AutoModelForSeq2SeqLM.from_pretrained(MODEL_CHECKPOINT)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"Trainable: {trainable:,} / {total:,} ({100*trainable/total:.1f}%)")

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/44.0 [00:00<?, ?B/s]

source.spm:   0%|          | 0.00/809k [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/756k [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/289M [00:00<?, ?B/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/293 [00:00<?, ?B/s]

Trainable: 127,122,944 / 127,122,944 (100.0%)


In [9]:
def preprocess_function(examples):
    model_inputs = tokenizer(
        examples[SOURCE_LANGUAGE],
        max_length=MAX_INPUT_LENGTH,
        truncation=True)
    
    labels = tokenizer(
        text_target=examples[TARGET_LANGUAGE],
        max_length=MAX_TARGET_LENGTH,
        truncation=True)
    
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tokenized_datasets = raw_datasets.map(
    preprocess_function,
    batched=True,
    remove_columns=raw_datasets["train"].column_names,
    desc="Tokenizing")

Tokenizing:   0%|          | 0/290000 [00:00<?, ? examples/s]

Tokenizing:   0%|          | 0/15000 [00:00<?, ? examples/s]

Tokenizing:   0%|          | 0/25000 [00:00<?, ? examples/s]

In [10]:
metric_bleu = evaluate.load("sacrebleu")
metric_chrf = evaluate.load("chrf")

In [11]:
def postprocess_text(preds, labels):
    preds  = [p.strip() for p in preds]
    labels = [[l.strip()] for l in labels]
    return preds, labels

In [12]:
def compute_metrics(eval_preds):
    preds, labels = eval_preds
    if isinstance(preds, tuple):
        preds = preds[0]

    # Replace any out-of-range / -100 sentinels before decoding (safety)
    preds  = np.where((preds >= 0) & (preds < tokenizer.vocab_size),
                      preds, tokenizer.pad_token_id)
    decoded_preds  = tokenizer.batch_decode(preds, skip_special_tokens=True)

    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    decoded_preds, decoded_labels = postprocess_text(decoded_preds, decoded_labels)

    bleu   = metric_bleu.compute(predictions=decoded_preds, references=decoded_labels)
    chrf   = metric_chrf.compute(predictions=decoded_preds, references=decoded_labels, word_order=0)
    chrfpp = metric_chrf.compute(predictions=decoded_preds, references=decoded_labels, word_order=2)

    # Gen length from decoded text (robust, avoids pad-counting issues)
    gen_len = np.mean([len(p.split()) for p in decoded_preds])

    return {
        "bleu":   round(bleu["score"], 4),
        "chrf":   round(chrf["score"], 4),
        "chrf++": round(chrfpp["score"], 4),
        "gen_len": round(float(gen_len), 4),}

In [13]:
data_collator = DataCollatorForSeq2Seq(tokenizer, model=model, padding="longest")


args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,

    eval_strategy="steps",
    save_strategy="steps",
    eval_steps=2000,
    save_steps=2000,
    save_total_limit=2,

    learning_rate=4e-5,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    weight_decay=0.01,

    # OOM-safe: small batch + accumulation = effective batch 32
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=2,
    eval_accumulation_steps=4,      # move eval logits to CPU in chunks
    group_by_length=True,           # less padding -> less memory & faster

    num_train_epochs=8,             # early stopping will likely cut this short

    predict_with_generate=True,
    generation_num_beams=2,         # cheaper eval; use 4 for final test
    generation_max_length=MAX_TARGET_LENGTH,

    fp16=True,                      

    logging_steps=200,
    report_to="none",

    load_best_model_at_end=True,
    metric_for_best_model="bleu",
    greater_is_better=True,)

trainer = Seq2SeqTrainer(
    model=model,
    args=args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


In [14]:
last_ckpt = None
if os.path.isdir(OUTPUT_DIR):
    ckpts = [d for d in os.listdir(OUTPUT_DIR) if d.startswith("checkpoint-")]
    if ckpts:
        last_ckpt = os.path.join(OUTPUT_DIR, sorted(ckpts, key=lambda x: int(x.split("-")[1]))[-1])
        print(f"Resuming from: {last_ckpt}")

trainer.train(resume_from_checkpoint=last_ckpt)

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Step,Training Loss,Validation Loss,Bleu,Chrf,Chrf++,Gen Len
2000,6.656989,1.485348,35.884800,53.383500,53.537800,23.518100
4000,6.219782,1.390784,38.178300,55.034900,55.220900,23.386500
6000,5.610541,1.346817,39.479200,56.134200,56.306500,23.650900
8000,5.565555,1.318632,40.122300,56.644300,56.813600,23.624100
10000,5.082015,1.302289,40.604000,57.079900,57.233900,23.670700
12000,5.091787,1.285638,40.931400,57.255600,57.420100,23.566100
14000,4.688553,1.279410,41.315200,57.579000,57.741600,23.524800
16000,4.742557,1.269814,41.570800,57.793800,57.944400,23.688800
18000,4.750714,1.261559,41.689600,57.853500,58.019000,23.499900
20000,4.488441,1.262115,41.809800,57.978700,58.138100,23.622200


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['model.encoder.embed_positions.weight', 'model.decoder.embed_positions.weight', 'lm_head.weight'].


TrainOutput(global_step=36256, training_loss=4.90938740542581, metrics={'train_runtime': 23702.0916, 'train_samples_per_second': 97.882, 'train_steps_per_second': 1.53, 'total_flos': 1.5920286579818496e+16, 'train_loss': 4.90938740542581, 'epoch': 8.0})

In [15]:
# ── Test set 1: MTet ───────────────────────────────────────────────────────
test_results = trainer.predict(
    tokenized_datasets["test"],
    num_beams=4,
    max_length=MAX_TARGET_LENGTH,
)
print("MTet TEST METRICS:", test_results.metrics)

# ── Test set 2: IELTS Academic ─────────────────────────────────────────────
ielts_raw = load_dataset("parquet", data_files={"test": IELTS_PARQUET_PATH})

ielts_tok = ielts_raw.map(
    preprocess_function,
    batched=True,
    remove_columns=ielts_raw["test"].column_names,
    desc="Tokenising IELTS",
)

ielts_results = trainer.predict(
    ielts_tok["test"],
    num_beams=4,
    max_length=MAX_TARGET_LENGTH,
)
print("IELTS Academic TEST METRICS:", ielts_results.metrics)

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


MTet TEST METRICS: {'test_loss': 1.2503421306610107, 'test_bleu': 42.4909, 'test_chrf': 58.6178, 'test_chrf++': 58.7567, 'test_gen_len': 23.5241, 'test_runtime': 866.877, 'test_samples_per_second': 28.839, 'test_steps_per_second': 0.902}


Generating test split: 0 examples [00:00, ? examples/s]

Tokenising IELTS:   0%|          | 0/6304 [00:00<?, ? examples/s]

IELTS Academic TEST METRICS: {'test_loss': 0.9986703395843506, 'test_bleu': 50.2315, 'test_chrf': 64.9431, 'test_chrf++': 64.9927, 'test_gen_len': 32.1967, 'test_runtime': 260.2118, 'test_samples_per_second': 24.226, 'test_steps_per_second': 0.757}


In [16]:
final_model_path = f"{OUTPUT_DIR}/final_model"
os.makedirs(final_model_path, exist_ok=True)

# Bake sensible generation defaults into the model for the app
model.generation_config.num_beams = 4
model.generation_config.max_length = MAX_TARGET_LENGTH
model.generation_config.early_stopping = True

trainer.save_model(final_model_path)
tokenizer.save_pretrained(final_model_path)
print(f"Saved inference model to: {final_model_path}")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved inference model to: /kaggle/working/marianmt-mtet-model-exp02/final_model
